# 4. Power BI Datamart (Polars)

This notebook builds the same Power BI tables using Polars for faster groupby and IO performance.

In [1]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks_polars':
    ROOT = ROOT.parent

RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
POWERBI_OUT = PROCESSED / 'powerbi'
PARQUET_CACHE = PROCESSED / 'parquet_cache'

POWERBI_OUT.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)

ROOT: C:\Users\sayee\Documents\Credit Default Risk


In [2]:
def load_table(name: str) -> pl.DataFrame:
    pq = PARQUET_CACHE / f'{name}.parquet'
    csv = RAW / f'{name}.csv'
    if pq.exists():
        return pl.read_parquet(pq)
    return pl.read_csv(csv, infer_schema_length=10000)


def safe_divide(numer: str, denom: str) -> pl.Expr:
    return pl.when(pl.col(denom).is_null() | (pl.col(denom) == 0))\
        .then(None)\
        .otherwise(pl.col(numer) / pl.col(denom))

In [4]:
app_train = load_table('application_train')
app_test = load_table('application_test').with_columns(pl.lit(None).alias('TARGET'))
bureau = load_table('bureau')
prev = load_table('previous_application')
inst = load_table('installments_payments')
pos = load_table('POS_CASH_balance')
cc = load_table('credit_card_balance')

app_train = app_train.with_columns(pl.lit('train').alias('DATASET'))
app_test = app_test.with_columns(pl.lit('test').alias('DATASET'))
app = pl.concat([app_train, app_test], how='diagonal_relaxed')

app = app.with_columns([
    pl.mean_horizontal(['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']).alias('EXT_SOURCE_MEAN')
])

median_ext = app.select(pl.col('EXT_SOURCE_MEAN').median()).item()
app = app.with_columns([
    (1.0 - pl.col('EXT_SOURCE_MEAN')).fill_null(1.0 - median_ext).clip(0.0, 1.0).alias('RISK_SCORE_BASE')
])

quantiles = app.select([
    pl.col('RISK_SCORE_BASE').quantile(0.2).alias('q20'),
    pl.col('RISK_SCORE_BASE').quantile(0.4).alias('q40'),
    pl.col('RISK_SCORE_BASE').quantile(0.6).alias('q60'),
    pl.col('RISK_SCORE_BASE').quantile(0.8).alias('q80'),
]).row(0)
q20, q40, q60, q80 = quantiles

app = app.with_columns([
    pl.when(pl.col('RISK_SCORE_BASE') <= q20).then(pl.lit('Very Low'))
    .when(pl.col('RISK_SCORE_BASE') <= q40).then(pl.lit('Low'))
    .when(pl.col('RISK_SCORE_BASE') <= q60).then(pl.lit('Medium'))
    .when(pl.col('RISK_SCORE_BASE') <= q80).then(pl.lit('High'))
    .otherwise(pl.lit('Very High'))
    .alias('RISK_BAND')
])

print('Combined application shape:', app.shape)

Combined application shape: (356255, 126)


In [5]:
dim_customer = app.select([
    'SK_ID_CURR',
    'CODE_GENDER',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_INCOME_TYPE',
    'NAME_HOUSING_TYPE',
    'REGION_RATING_CLIENT',
    'CNT_CHILDREN',
    'CNT_FAM_MEMBERS',
    'DAYS_BIRTH',
    'DAYS_EMPLOYED',
    'AMT_INCOME_TOTAL',
]).with_columns([
    ((-pl.col('DAYS_BIRTH')) / 365).round(1).alias('AGE_YEARS'),
    pl.when(pl.col('DAYS_EMPLOYED') < 0)
    .then(((-pl.col('DAYS_EMPLOYED')) / 365).round(1))
    .otherwise(None)
    .alias('EMPLOYMENT_YEARS'),
])

fact_application = app.select([
    'SK_ID_CURR',
    'DATASET',
    'TARGET',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'AMT_GOODS_PRICE',
    'AMT_INCOME_TOTAL',
    'NAME_CONTRACT_TYPE',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'RISK_SCORE_BASE',
    'RISK_BAND',
]).with_columns([
    pl.col('RISK_SCORE_BASE').alias('BASE_RISK_SCORE'),
    pl.col('RISK_BAND').alias('BASE_RISK_BAND'),
])

In [6]:
bureau_fact = bureau.group_by('SK_ID_CURR').agg([
    pl.len().alias('BUREAU_LOAN_COUNT'),
    (pl.col('CREDIT_ACTIVE') == 'Active').sum().alias('BUREAU_ACTIVE_COUNT'),
    (pl.col('CREDIT_ACTIVE') == 'Closed').sum().alias('BUREAU_CLOSED_COUNT'),
    pl.col('AMT_CREDIT_SUM_DEBT').sum().alias('BUREAU_DEBT_SUM'),
    pl.col('AMT_CREDIT_SUM_OVERDUE').sum().alias('BUREAU_OVERDUE_SUM'),
    pl.col('AMT_CREDIT_SUM').sum().alias('BUREAU_CREDIT_SUM'),
])

prev_fact = prev.group_by('SK_ID_CURR').agg([
    pl.len().alias('PREV_APP_COUNT'),
    (pl.col('NAME_CONTRACT_STATUS') == 'Approved').sum().alias('PREV_APPROVED_COUNT'),
    (pl.col('NAME_CONTRACT_STATUS') == 'Refused').sum().alias('PREV_REFUSED_COUNT'),
    pl.col('AMT_APPLICATION').sum().alias('PREV_AMT_APPLICATION_SUM'),
    pl.col('AMT_CREDIT').sum().alias('PREV_AMT_CREDIT_SUM'),
])

inst2 = inst.with_columns([
    (pl.col('DAYS_ENTRY_PAYMENT') - pl.col('DAYS_INSTALMENT')).alias('DAYS_LATE'),
    (pl.col('DAYS_ENTRY_PAYMENT') - pl.col('DAYS_INSTALMENT') > 0).cast(pl.Int8).alias('IS_LATE'),
])
installments_fact = inst2.group_by('SK_ID_CURR').agg([
    pl.len().alias('INSTAL_COUNT'),
    pl.col('DAYS_LATE').mean().alias('INST_DAYS_LATE_MEAN'),
    pl.col('DAYS_LATE').max().alias('INST_DAYS_LATE_MAX'),
    pl.col('IS_LATE').mean().alias('INST_LATE_RATE'),
    pl.col('AMT_PAYMENT').sum().alias('INST_PAYMENT_SUM'),
])

pos_fact = pos.group_by('SK_ID_CURR').agg([
    pl.len().alias('POS_RECORD_COUNT'),
    pl.col('SK_DPD').mean().alias('POS_DPD_MEAN'),
    pl.col('SK_DPD').max().alias('POS_DPD_MAX'),
    pl.col('SK_DPD_DEF').mean().alias('POS_DPD_DEF_MEAN'),
    pl.col('SK_DPD_DEF').max().alias('POS_DPD_DEF_MAX'),
])

cc2 = cc.with_columns([
    safe_divide('AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL').alias('CC_UTILIZATION')
])
cc_fact = cc2.group_by('SK_ID_CURR').agg([
    pl.len().alias('CC_RECORD_COUNT'),
    pl.col('AMT_BALANCE').sum().alias('CC_BALANCE_SUM'),
    pl.col('AMT_DRAWINGS_CURRENT').sum().alias('CC_DRAWINGS_SUM'),
    pl.col('AMT_PAYMENT_TOTAL_CURRENT').sum().alias('CC_PAYMENT_SUM'),
    pl.col('CC_UTILIZATION').mean().alias('CC_UTILIZATION_MEAN'),
    pl.col('CC_UTILIZATION').max().alias('CC_UTILIZATION_MAX'),
])

In [7]:
risk_segment = fact_application.group_by('RISK_BAND').agg([
    pl.len().alias('CUSTOMER_COUNT'),
    pl.col('TARGET').fill_null(0).sum().alias('DEFAULT_COUNT'),
    pl.col('AMT_CREDIT').mean().alias('AVG_CREDIT'),
    pl.col('AMT_INCOME_TOTAL').mean().alias('AVG_INCOME'),
]).with_columns([
    safe_divide('DEFAULT_COUNT', 'CUSTOMER_COUNT').alias('DEFAULT_RATE')
]).rename({
    'RISK_BAND': 'BASE_RISK_BAND'
})

kpi_snapshot = pl.DataFrame({
    'TOTAL_CUSTOMERS': [fact_application.select(pl.col('SK_ID_CURR').n_unique()).item()],
    'TRAIN_CUSTOMERS': [fact_application.filter(pl.col('DATASET') == 'train').height],
    'TEST_CUSTOMERS': [fact_application.filter(pl.col('DATASET') == 'test').height],
    'DEFAULT_COUNT': [fact_application.select(pl.col('TARGET').fill_null(0).sum()).item()],
    'DEFAULT_RATE': [fact_application.select(pl.col('TARGET').mean()).item()],
    'TOTAL_CREDIT_AMOUNT': [fact_application.select(pl.col('AMT_CREDIT').sum()).item()],
    'AVG_CREDIT_AMOUNT': [fact_application.select(pl.col('AMT_CREDIT').mean()).item()],
    'AVG_ANNUITY': [fact_application.select(pl.col('AMT_ANNUITY').mean()).item()],
    'AVG_INCOME': [fact_application.select(pl.col('AMT_INCOME_TOTAL').mean()).item()],
})

outputs = {
    'dim_customer_polars': dim_customer,
    'fact_application_polars': fact_application,
    'fact_bureau_polars': bureau_fact,
    'fact_previous_application_polars': prev_fact,
    'fact_installments_polars': installments_fact,
    'fact_pos_cash_polars': pos_fact,
    'fact_credit_card_polars': cc_fact,
    'dim_risk_segment_polars': risk_segment,
    'kpi_snapshot_polars': kpi_snapshot,
}

for name, df in outputs.items():
    df.write_parquet(POWERBI_OUT / f'{name}.parquet')
    df.write_csv(POWERBI_OUT / f'{name}.csv')

print('Polars Power BI datamart generated:')
for name, df in outputs.items():
    print(f'- {name}: {df.shape}')

Polars Power BI datamart generated:
- dim_customer_polars: (356255, 14)
- fact_application_polars: (356255, 12)
- fact_bureau_polars: (305811, 7)
- fact_previous_application_polars: (338857, 6)
- fact_installments_polars: (339587, 6)
- fact_pos_cash_polars: (337252, 6)
- fact_credit_card_polars: (103558, 7)
- dim_risk_segment_polars: (5, 6)
- kpi_snapshot_polars: (1, 9)
